# COVID-19 TCR Biomarker × HLA Allele Associations

Test whether HLA alleles are associated with the presence of COVID-19 enriched
TCR clonotypes discovered in `covid19_biomarkers.ipynb`.  
For each significant clonotype × HLA allele pair we run a 2×2 Fisher test across
1137 paired donors from the `isalgo/airr_covid19` cohort.

**Data sources**
- `tmp/fisher_trb.parquet` / `tmp/fisher_tra.parquet` — de-novo Fisher scan results  
- `notebooks/assets/large/airr_covid19/metadata.tsv` — donor metadata with HLA columns  
- AIRR TSV files — read to get per-donor CDR3 presence for significant clonotypes

## 1. Setup

In [ ]:
from __future__ import annotations
import json, os, sys, time
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.stats import fisher_exact
from statsmodels.stats.multitest import multipletests
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

SEED = 42
np.random.seed(SEED)

def find_repo_root(start=None):
    here = (start or Path.cwd()).resolve()
    for cand in (here, *here.parents):
        if (cand / 'pyproject.toml').exists() and (cand / 'mir').exists():
            return cand
    raise FileNotFoundError('Cannot find repo root')

repo_root    = find_repo_root()
data_root    = repo_root / 'notebooks' / 'assets' / 'large' / 'airr_covid19'
tmp_dir      = repo_root / 'tmp'
assets_dir   = repo_root / 'notebooks' / 'assets'

import importlib.metadata
print(f'Python {sys.version.split()[0]}  mirpy-lib {importlib.metadata.version("mirpy-lib")}')
print(f'numpy {np.__version__}  pandas {pd.__version__}')
print(f'data_root exists: {data_root.exists()}')

## 2. Load Significant Biomarker Clonotypes

In [ ]:
# Load pre-computed Fisher results (from covid19_biomarkers.ipynb scan).
FDR_THRESHOLD = 0.05

fisher_trb = pd.read_parquet(tmp_dir / 'fisher_trb.parquet')
fisher_tra = pd.read_parquet(tmp_dir / 'fisher_tra.parquet')

sig_trb = fisher_trb[fisher_trb['p_value_adj'] < FDR_THRESHOLD][['junction_aa','locus','log2_fe','p_value_adj']].copy()
sig_tra = fisher_tra[fisher_tra['p_value_adj'] < FDR_THRESHOLD][['junction_aa','locus','log2_fe','p_value_adj']].copy()
sig_all = pd.concat([sig_trb, sig_tra], ignore_index=True)

# Separate COVID-enriched and healthy-enriched for display
sig_covid   = sig_all[sig_all['log2_fe'] > 0]
sig_healthy = sig_all[sig_all['log2_fe'] < 0]

print(f'Significant biomarkers loaded:')
print(f'  TRB: {len(sig_trb)}  (COVID-enriched: {(sig_trb.log2_fe>0).sum()}, healthy-enriched: {(sig_trb.log2_fe<0).sum()})')
print(f'  TRA: {len(sig_tra)}  (COVID-enriched: {(sig_tra.log2_fe>0).sum()}, healthy-enriched: {(sig_tra.log2_fe<0).sum()})')
print(f'  Total: {len(sig_all)}')
print(sig_all[['junction_aa','locus','log2_fe','p_value_adj']].to_string(index=False))

## 3. Load Donor Metadata and HLA Alleles

In [ ]:
# Load metadata and build one-row-per-donor HLA table.
meta = pd.read_csv(data_root / 'metadata.tsv', sep='\t', dtype={'donor_id': 'string'})
HLA_COLS = [c for c in meta.columns if c.startswith('HLA')]

meta_filt = meta[meta['COVID_status'].isin(['COVID', 'healthy'])].copy()
meta_filt['locus_upper'] = meta_filt['locus'].str.upper()
trb_donors = set(meta_filt[meta_filt['locus_upper']=='TRB']['donor_id'])
tra_donors = set(meta_filt[meta_filt['locus_upper']=='TRA']['donor_id'])
paired_donors = sorted(trb_donors & tra_donors)

# One row per donor (deduplication; TRB row carries HLA typing)
meta_donor = (
    meta_filt[meta_filt['donor_id'].isin(paired_donors)]
    .sort_values('locus_upper')                     # TRA first, TRB second
    .drop_duplicates(subset='donor_id', keep='last') # keep TRB row
    .set_index('donor_id')
    .copy()
)

n_covid   = (meta_donor['COVID_status'] == 'COVID').sum()
n_healthy = (meta_donor['COVID_status'] == 'healthy').sum()
print(f'Paired donors: {len(meta_donor)}  COVID={n_covid}  healthy={n_healthy}')
print(f'HLA columns ({len(HLA_COLS)}): {HLA_COLS}')

# Build per-donor × HLA-allele binary presence matrix.
# Melt _1/_2 allele columns per gene: donor 'has allele' if either allele matches.
hla_genes = sorted({c.rsplit('_', 1)[0] for c in HLA_COLS})
hla_presence_rows = []

for donor_id, row in meta_donor.iterrows():
    alleles_seen = set()
    for gene in hla_genes:
        for suffix in ['_1', '_2']:
            col = gene + suffix
            val = row.get(col)
            if pd.notna(val) and str(val).strip():
                alleles_seen.add(str(val).strip())
    hla_presence_rows.append({'donor_id': donor_id, 'alleles': alleles_seen,
                              'COVID_status': row['COVID_status']})

# Collect all observed alleles with frequency ≥ 5% donors (for statistical power)
from collections import Counter
allele_counts = Counter(a for r in hla_presence_rows for a in r['alleles'])
MIN_ALLELE_FREQ = 0.05
min_allele_n    = int(len(hla_presence_rows) * MIN_ALLELE_FREQ)
common_alleles  = sorted(a for a, n in allele_counts.items() if n >= min_allele_n)

print(f'Total unique HLA alleles: {len(allele_counts)}')
print(f'Alleles with ≥{MIN_ALLELE_FREQ*100:.0f}% donors (n≥{min_allele_n}): {len(common_alleles)}')
print('Top 10 alleles:', sorted(allele_counts.items(), key=lambda x: -x[1])[:10])

## 4. Per-Donor CDR3 Presence for Significant Clonotypes

In [ ]:
# Scan each donor file for presence of significant CDR3s (CDR3 column only).
# Split by locus to use correct file (TRA file for TRA clonotypes, etc.).
_STOP = frozenset('*_X')
sig_trb_set = set(sig_trb['junction_aa'].astype(str))
sig_tra_set = set(sig_tra['junction_aa'].astype(str))
sig_sets    = {'TRB': sig_trb_set, 'TRA': sig_tra_set}

meta_by_locus = {
    'TRB': meta_filt[meta_filt['locus_upper']=='TRB'].set_index('donor_id'),
    'TRA': meta_filt[meta_filt['locus_upper']=='TRA'].set_index('donor_id'),
}

# donor_id -> {cdr3aa} presence dict per significant CDR3
donor_cdr3_hit: dict[str, set[str]] = {d: set() for d in paired_donors}

t0 = time.perf_counter()
for locus_key, target_set in sig_sets.items():
    if not target_set:
        continue
    locus_meta = meta_by_locus[locus_key]
    for donor_id in paired_donors:
        if donor_id not in locus_meta.index:
            continue
        path = data_root / str(locus_meta.loc[donor_id, 'file_name'])
        try:
            df = pd.read_csv(path, sep='\t', usecols=['cdr3aa'], dtype=str)
        except Exception:
            continue
        hits = set(df['cdr3aa'].dropna()) & target_set
        donor_cdr3_hit[donor_id].update(hits)

elapsed = time.perf_counter() - t0
n_with_any_hit = sum(1 for v in donor_cdr3_hit.values() if v)
print(f'CDR3 presence scan: {elapsed:.1f}s')
print(f'Donors with ≥1 sig hit: {n_with_any_hit}/{len(paired_donors)}')
for cdr3 in sorted(sig_trb_set | sig_tra_set):
    n = sum(1 for v in donor_cdr3_hit.values() if cdr3 in v)
    print(f'  {cdr3:30s}  n={n}')

## 5. Fisher Test: HLA Allele × Clonotype Presence

In [ ]:
# For each (significant CDR3, common HLA allele): 2×2 Fisher test.
# Table: [[has_allele & has_cdr3, has_allele & no_cdr3],
#          [no_allele  & has_cdr3, no_allele  & no_cdr3]]
sig_cdr3_list  = sorted(sig_trb_set | sig_tra_set)

rows = []
for allele in common_alleles:
    # donors with this allele
    has_allele = {r['donor_id'] for r in hla_presence_rows if allele in r['alleles']}
    no_allele  = set(paired_donors) - has_allele

    for cdr3 in sig_cdr3_list:
        has_both   = sum(1 for d in has_allele if cdr3 in donor_cdr3_hit.get(d, set()))
        allele_only= len(has_allele) - has_both
        cdr3_only  = sum(1 for d in no_allele  if cdr3 in donor_cdr3_hit.get(d, set()))
        neither    = len(no_allele)  - cdr3_only

        if has_both + cdr3_only == 0 or has_both + allele_only == 0:
            continue  # clonotype or allele absent in one group — skip

        _, p = fisher_exact([[has_both, allele_only], [cdr3_only, neither]],
                            alternative='two-sided')

        eps = 0.5
        n_allele = len(has_allele)
        n_no     = len(no_allele)
        f_with   = (has_both + eps) / (n_allele + 2 * eps)
        f_without= (cdr3_only + eps) / (n_no + 2 * eps)
        fe       = f_with / f_without

        rows.append({
            'hla_allele': allele,
            'junction_aa': cdr3,
            'locus': 'TRB' if cdr3 in sig_trb_set else 'TRA',
            'n_allele': n_allele,
            'n_has_both': has_both,
            'n_cdr3_no_allele': cdr3_only,
            'freq_with_allele': f_with,
            'freq_without_allele': f_without,
            'fold_enrichment': fe,
            'log2_fe': float(np.log2(max(fe, 1e-6))),
            'p_value': p,
        })

hla_df = pd.DataFrame(rows)
if not hla_df.empty:
    _, padj, _, _ = multipletests(hla_df['p_value'], method='fdr_bh')
    hla_df['p_value_adj'] = padj
    hla_df['neg_log10_padj'] = -np.log10(np.clip(padj, 1e-300, None))
    hla_df = hla_df.sort_values(['p_value_adj', 'p_value']).reset_index(drop=True)

n_sig = int((hla_df['p_value_adj'] < FDR_THRESHOLD).sum()) if not hla_df.empty else 0
print(f'HLA-clonotype pairs tested: {len(hla_df)}')
print(f'Significant (q<{FDR_THRESHOLD}): {n_sig}')
print(hla_df.head(20)[['hla_allele','junction_aa','locus','n_allele','n_has_both',
                         'log2_fe','p_value','p_value_adj']].to_string(index=False))

## 6. Volcano Plot and Heatmap

In [ ]:
if hla_df.empty:
    print('No data to plot.')
else:
    sns.set_theme(style='whitegrid', context='talk')

    # ── Volcano ──────────────────────────────────────────────────────────────
    sig_mask = hla_df['p_value_adj'] < FDR_THRESHOLD
    colors = np.where(sig_mask & (hla_df['log2_fe'] > 0), '#e63946',
             np.where(sig_mask & (hla_df['log2_fe'] < 0), '#457b9d', '#adb5bd'))

    fig, ax = plt.subplots(figsize=(10, 6))
    ax.scatter(hla_df['log2_fe'], hla_df['neg_log10_padj'],
               c=colors, s=18, alpha=0.7, linewidths=0)
    ax.axhline(-np.log10(FDR_THRESHOLD), color='#6c757d', ls='--', lw=1, label=f'q={FDR_THRESHOLD}')
    ax.axvline(0, color='#6c757d', ls=':', lw=0.8)

    top = hla_df[sig_mask].head(15)
    for _, row in top.iterrows():
        ax.text(float(row['log2_fe']) + 0.05, float(row['neg_log10_padj']),
                f"{row['hla_allele']} / {row['junction_aa']}", fontsize=6, va='center')

    ax.set_xlabel('log₂(fold enrichment  with_allele / without_allele)')
    ax.set_ylabel('−log₁₀(q)')
    ax.set_title(f'HLA Allele × TCR Biomarker Association\n'
                 f'(sig={n_sig} of {len(hla_df)} tested pairs, q<{FDR_THRESHOLD})',
                 fontweight='bold')
    ax.legend(fontsize=9)
    plt.tight_layout()
    plt.savefig(assets_dir / 'covid19_hla_volcano.pdf', bbox_inches='tight')
    plt.show()

    # ── Heatmap of top associations ───────────────────────────────────────────
    if n_sig > 0:
        top_pairs = hla_df[sig_mask][['hla_allele','junction_aa','log2_fe']].copy()
        pivot = top_pairs.pivot_table(index='junction_aa', columns='hla_allele',
                                       values='log2_fe', aggfunc='first')
        fig2, ax2 = plt.subplots(figsize=(max(6, pivot.shape[1]*0.6 + 2),
                                          max(4, pivot.shape[0]*0.5 + 2)))
        sns.heatmap(pivot, cmap='RdBu_r', center=0, ax=ax2,
                    linewidths=0.5, cbar_kws={'label': 'log₂FE'})
        ax2.set_title('Significant HLA × Clonotype log₂FE', fontweight='bold')
        ax2.set_xlabel('HLA allele');  ax2.set_ylabel('Biomarker CDR3')
        plt.tight_layout()
        plt.savefig(assets_dir / 'covid19_hla_heatmap.pdf', bbox_inches='tight')
        plt.show()
    else:
        print('No significant pairs after FDR correction — no heatmap produced.')
        print('Top nominal p-value pairs:')
        print(hla_df.head(10)[['hla_allele','junction_aa','locus',
                                'log2_fe','p_value','p_value_adj']].to_string(index=False))

## 7. Summary

In [ ]:
print('=== HLA × TCR Biomarker Association Summary ===')
print(f'  Significant clonotypes tested: {len(sig_cdr3_list)}')
print(f'  Common HLA alleles tested (≥5%): {len(common_alleles)}')
print(f'  Total (clonotype × allele) pairs: {len(hla_df)}')
print(f'  Significant after BH correction (q<{FDR_THRESHOLD}): {n_sig}')
if n_sig > 0:
    print('\nTop significant pairs:')
    sig_top = hla_df[hla_df['p_value_adj'] < FDR_THRESHOLD].head(20)
    print(sig_top[['hla_allele','junction_aa','locus','n_has_both',
                    'log2_fe','p_value_adj']].to_string(index=False))
    hla_df[hla_df['p_value_adj'] < FDR_THRESHOLD].to_csv(
        repo_root / 'tmp' / 'hla_sig_pairs.csv', index=False)
    print('Significant pairs saved to tmp/hla_sig_pairs.csv')
else:
    print('  → No statistically significant HLA-clonotype associations found.')
    print('  → Consider: (a) increase sample size, (b) relaxed q threshold,'
          ' (c) broader allele grouping (gene family instead of allele).')